In [1]:
import pandas as pd
path = "https://raw.githubusercontent.com/ulka29/paper/249de2a1bc6c07516904de4a27993d56eaa9e2ad/01_data/processed/df_cleaned_ver2.csv"# import Check dataset
df= pd.read_csv(path)
df.head()

,tweet_text,cyberbullying_type,clean_tweet,lemmatized_tweet,sentiment_scores,negative,positive,neutral,compound
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr,word katandandre food crapilicious mkr,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...,aussietv white mkr theblock imacelebrityau tod...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes,classy whore red velvet cupcake,"{'neg': 0.32, 'neu': 0.481, 'pos': 0.199, 'com...",0.320,0.199,0.481,-0.4137
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...,meh p thanks head concerned another angry dude...,"{'neg': 0.243, 'neu': 0.607, 'pos': 0.15, 'com...",0.243,0.150,0.607,-0.4854
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...,isi account pretending kurdish account like is...,"{'neg': 0.129, 'neu': 0.691, 'pos': 0.18, 'com...",0.129,0.180,0.691,0.0258


In [2]:
# Select relevant features for SA + ML
df_ml = df[
    [
        'lemmatized_tweet',
        'negative',
        'positive',
        'neutral',
        'compound',
        'cyberbullying_type'
    ]
].copy()

# Check shape
print("\n:-------------Shape::-------------", df_ml.shape)

# Check column names
print("\n:-------------Columns::-------------")
print(df_ml.columns.tolist())

# Check missing values
print("\n----------------Missing values:-------------")
print(df_ml.isnull().sum())

# Display first 5 rows
df_ml.head()


:-------------Shape::------------- (42602, 6)

:-------------Columns::-------------
['lemmatized_tweet', 'negative', 'positive', 'neutral', 'compound', 'cyberbullying_type']

----------------Missing values:-------------
lemmatized_tweet      0
negative              0
positive              0
neutral               0
compound              0
cyberbullying_type    0
dtype: int64


,lemmatized_tweet,negative,positive,neutral,compound,cyberbullying_type
0,word katandandre food crapilicious mkr,0.000,0.000,1.000,0.0000,not_cyberbullying
1,aussietv white mkr theblock imacelebrityau tod...,0.000,0.000,1.000,0.0000,not_cyberbullying
2,classy whore red velvet cupcake,0.320,0.199,0.481,-0.4137,not_cyberbullying
3,meh p thanks head concerned another angry dude...,0.243,0.150,0.607,-0.4854,not_cyberbullying
4,isi account pretending kurdish account like is...,0.129,0.180,0.691,0.0258,not_cyberbullying


In [3]:
PROFANITY_WORDS = {
    'fuck', 'fucking', 'fucked', 'fuckin',
    'shit', 'shitty',
    'bitch', 'bitches',
    'asshole', 'assholes',
    'bastard',
    'damn',
    'dick',
    'piss',
    'crap',
    'bullshit',
    'motherfucker',
    'idiot',
    'stupid',
    'moron',
    'retard',
    'retarded'
}

In [4]:
def extract_linguistic_features(text):

    text = str(text)

    # Basic tokens
    words = re.findall(r'\b\w+\b', text.lower())

    word_count = len(words)
    char_count = len(text)
    unique_words = len(set(words))

    # Profanity
    profanity_words = [
        word for word in words
        if word in PROFANITY_WORDS
    ]

    profanity_count = len(profanity_words)
    unique_profanity_count = len(set(profanity_words))

    profanity_ratio = (
        profanity_count / word_count
        if word_count > 0 else 0
    )

    profanity_presence = int(profanity_count > 0)

    # Second-person language
    second_person_words = {
        'you',
        'your',
        'yours',
        'yourself',
        'yourselves',
        'youre',
        'youve',
        'youll',
        'youd'
    }

    second_person_count = sum(
        word in second_person_words
        for word in words
    )

    # Punctuation
    exclamation_count = text.count('!')
    question_count = text.count('?')
    comma_count = text.count(',')
    period_count = text.count('.')

    repeated_exclamation_count = len(
        re.findall(r'!{2,}', text)
    )

    repeated_question_count = len(
        re.findall(r'\?{2,}', text)
    )

    repeated_punctuation_count = len(
        re.findall(r'[!?]{2,}', text)
    )

    # Mentions / Hashtags
    mention_count = len(
        re.findall(r'@\w+', text)
    )

    hashtag_count = len(
        re.findall(r'#\w+', text)
    )

    # Return features
    return pd.Series({
        'word_count': word_count,
        'char_count': char_count,
        'unique_word_count': unique_words,

        'profanity_count': profanity_count,
        'unique_profanity_count': unique_profanity_count,
        'profanity_ratio': profanity_ratio,
        'profanity_presence': profanity_presence,

        'second_person_count': second_person_count,

        'exclamation_count': exclamation_count,
        'question_count': question_count,
        'comma_count': comma_count,
        'period_count': period_count,

        'repeated_exclamation_count': repeated_exclamation_count,
        'repeated_question_count': repeated_question_count,
        'repeated_punctuation_count': repeated_punctuation_count,

        'mention_count': mention_count,
        'hashtag_count': hashtag_count
    })


In [5]:
import re

In [6]:
linguistic_features = df['tweet_text'].apply(
    extract_linguistic_features
)

df = pd.concat(
    [df, linguistic_features],
    axis=1
)

In [7]:
df.head()

,tweet_text,cyberbullying_type,clean_tweet,lemmatized_tweet,sentiment_scores,negative,positive,neutral,compound,word_count,...,second_person_count,exclamation_count,question_count,comma_count,period_count,repeated_exclamation_count,repeated_question_count,repeated_punctuation_count,mention_count,hashtag_count
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr,word katandandre food crapilicious mkr,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000,9.0,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...,aussietv white mkr theblock imacelebrityau tod...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000,14.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes,classy whore red velvet cupcake,"{'neg': 0.32, 'neu': 0.481, 'pos': 0.199, 'com...",0.320,0.199,0.481,-0.4137,9.0,...,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...,meh p thanks head concerned another angry dude...,"{'neg': 0.243, 'neu': 0.607, 'pos': 0.15, 'com...",0.243,0.150,0.607,-0.4854,18.0,...,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,1.0,0.0
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...,isi account pretending kurdish account like is...,"{'neg': 0.129, 'neu': 0.691, 'pos': 0.18, 'com...",0.129,0.180,0.691,0.0258,18.0,...,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,1.0,0.0


In [8]:
print("Class-wise counts:")
for class_name, count in df['cyberbullying_type'].value_counts().items():
    print(f"{class_name}: {count}")

Class-wise counts:
religion: 7963
age: 7959
gender: 7478
ethnicity: 7426
not_cyberbullying: 6435
other_cyberbullying: 5341


In [9]:
print(df.columns.tolist())

['tweet_text', 'cyberbullying_type', 'clean_tweet', 'lemmatized_tweet', 'sentiment_scores', 'negative', 'positive', 'neutral', 'compound', 'word_count', 'char_count', 'unique_word_count', 'profanity_count', 'unique_profanity_count', 'profanity_ratio', 'profanity_presence', 'second_person_count', 'exclamation_count', 'question_count', 'comma_count', 'period_count', 'repeated_exclamation_count', 'repeated_question_count', 'repeated_punctuation_count', 'mention_count', 'hashtag_count']


In [10]:
# from google.colab import files

# df.to_csv('engineered_feature_dataset.csv', index=False)
# files.download('engineered_feature_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# ============================================================
# CELL 1: DATA PREPARATION
# HEXA CLASS + WORD TF-IDF + SENTIMENT FEATURES
# DATASET: df_ml
# ============================================================

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# ------------------------------------------------------------
# 1. SELECT TEXT, SENTIMENT FEATURES AND TARGET
# ------------------------------------------------------------

text_col = "lemmatized_tweet"
target_col = "cyberbullying_type"

sentiment_features = [
    "negative",
    "positive",
    "neutral",
    "compound"
]

behavioral_features = [
    "second_person_count",
    "mention_count",
    "hashtag_count",
    "profanity_count",
    "unique_profanity_count",
    "profanity_ratio",
    "profanity_presence"
]

# Keep rows with valid target labels
df_exp = df.dropna(subset=[target_col]).copy()

X_text = df_exp[text_col].fillna("").astype(str)

# Convert numeric features safely
all_numeric_features = sentiment_features + behavioral_features

X_numeric = (
    df_exp[all_numeric_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = df_exp[target_col].astype(str)

# ------------------------------------------------------------
# 2. LABEL ENCODING
# ------------------------------------------------------------

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Classes:")
print(label_encoder.classes_)

label_mapping = dict(enumerate(label_encoder.classes_))

print("\nLabel Mapping:")
for number, category in label_mapping.items():
    print(f"{number} -> {category}")

print("\nDataset shape:", df_exp.shape)
print("Number of classes:", len(label_encoder.classes_))

# ------------------------------------------------------------
# 3. FIXED STRATIFIED TRAIN-TEST SPLIT
# ------------------------------------------------------------

(
    X_text_train,
    X_text_test,
    X_num_train,
    X_num_test,
    y_train,
    y_test
) = train_test_split(
    X_text,
    X_numeric,
    y_encoded,
    test_size=0.30,
    random_state=42,
    stratify=y_encoded
)

print("\nTrain-Test Split")
print("-------------------------")
print("Training samples:", len(y_train))
print("Testing samples :", len(y_test))

# ------------------------------------------------------------
# 4. WORD TF-IDF VECTORIZATION
# Fit ONLY on training text
# ------------------------------------------------------------

tfidf = TfidfVectorizer(
    analyzer="word",
    max_features=10000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_text_train)
X_test_tfidf = tfidf.transform(X_text_test)

print("\nWord TF-IDF Features")
print("-------------------------")
print("X_train TF-IDF:", X_train_tfidf.shape)
print("X_test TF-IDF :", X_test_tfidf.shape)



# ------------------------------------------------------------
# 4. SCALE NUMERIC FEATURES INDEPENDENTLY
# Fit scaler ONLY on training data
# ------------------------------------------------------------

scaler = StandardScaler()

# Fit on training data; transform train and test separately
X_num_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_num_train),
    columns=X_num_train.columns,
    index=X_num_train.index
)

X_num_test_scaled = pd.DataFrame(
    scaler.transform(X_num_test),
    columns=X_num_test.columns,
    index=X_num_test.index
)

print("\nNumeric Feature Scaling")
print("-------------------------")
print("Training means (approximately 0):")
print(X_num_train_scaled.mean().round(4))

print("\nTraining standard deviations (approximately 1):")
print(X_num_train_scaled.std(ddof=0).round(4))

Classes:
['age' 'ethnicity' 'gender' 'not_cyberbullying' 'other_cyberbullying'
 'religion']

Label Mapping:
0 -> age
1 -> ethnicity
2 -> gender
3 -> not_cyberbullying
4 -> other_cyberbullying
5 -> religion

Dataset shape: (42602, 26)
Number of classes: 6

Train-Test Split
-------------------------
Training samples: 29821
Testing samples : 12781

Word TF-IDF Features
-------------------------
X_train TF-IDF: (29821, 10000)
X_test TF-IDF : (12781, 10000)

Numeric Feature Scaling
-------------------------
Training means (approximately 0):
negative                 -0.0
positive                 -0.0
neutral                   0.0
compound                  0.0
second_person_count      -0.0
mention_count            -0.0
hashtag_count            -0.0
profanity_count           0.0
unique_profanity_count   -0.0
profanity_ratio          -0.0
profanity_presence       -0.0
dtype: float64

Training standard deviations (approximately 1):
negative                  1.0
positive                  1.0
neut

In [12]:
# ============================================================
# CELL 2: FEATURE CONFIGURATIONS
# EXPERIMENTS E1-E4
# ============================================================

# ------------------------------------------------------------
# 1. DEFINE EXPERIMENTS
# ------------------------------------------------------------

experiments = {
    "E1_Text_Sentiment": [
        "negative",
        "positive",
        "neutral",
        "compound"
    ],

    "E2_Sentiment_Profanity_Behavior": [
        "negative",
        "positive",
        "neutral",
        "compound",
        "second_person_count",
        "profanity_count",
        "unique_profanity_count",
        "profanity_ratio",
        "profanity_presence"
    ],

    "E3_Sentiment_Mention_Hashtag": [
        "negative",
        "positive",
        "neutral",
        "compound",
        "mention_count",
        "hashtag_count"
    ],

    "E4_Sentiment_All_Behavior": [
        "negative",
        "positive",
        "neutral",
        "compound",
        "second_person_count",
        "mention_count",
        "hashtag_count",
        "profanity_count",
        "unique_profanity_count",
        "profanity_ratio",
        "profanity_presence"
    ]
}

# ------------------------------------------------------------
# 2. CREATE COMBINED FEATURE MATRICES
# Same split and same TF-IDF representation for every experiment
# ------------------------------------------------------------

X_train_experiments = {}
X_test_experiments = {}

for exp_name, feature_cols in experiments.items():

    # Select the experiment's numeric features
    # Use scaled numeric features for every experiment
    X_num_train_exp = X_num_train_scaled[feature_cols]
    X_num_test_exp = X_num_test_scaled[feature_cols]

    # X_num_train_exp = X_num_train[feature_cols]
    # X_num_test_exp = X_num_test[feature_cols]

    # Convert numeric features to sparse matrices
    X_num_train_sparse = csr_matrix(
        X_num_train_exp.to_numpy(dtype=float)
    )

    X_num_test_sparse = csr_matrix(
        X_num_test_exp.to_numpy(dtype=float)
    )

    # Combine Word TF-IDF + selected numeric features
    X_train_experiments[exp_name] = hstack(
        [X_train_tfidf, X_num_train_sparse],
        format="csr"
    )

    X_test_experiments[exp_name] = hstack(
        [X_test_tfidf, X_num_test_sparse],
        format="csr"
    )

    print("\n" + "=" * 60)
    print(exp_name)
    print("=" * 60)
    print("Additional numeric features:", len(feature_cols) - 4)
    print("Total numeric features:", len(feature_cols))
    print("Training shape:", X_train_experiments[exp_name].shape)
    print("Testing shape :", X_test_experiments[exp_name].shape)


E1_Text_Sentiment
Additional numeric features: 0
Total numeric features: 4
Training shape: (29821, 10004)
Testing shape : (12781, 10004)

E2_Sentiment_Profanity_Behavior
Additional numeric features: 5
Total numeric features: 9
Training shape: (29821, 10009)
Testing shape : (12781, 10009)

E3_Sentiment_Mention_Hashtag
Additional numeric features: 2
Total numeric features: 6
Training shape: (29821, 10006)
Testing shape : (12781, 10006)

E4_Sentiment_All_Behavior
Additional numeric features: 7
Total numeric features: 11
Training shape: (29821, 10011)
Testing shape : (12781, 10011)


In [13]:
# ============================================================
# CELL 3: HEXA CLASS + WORD TF-IDF + SENTIMENT / BEHAVIORAL
# NO BALANCING + LOGISTIC REGRESSION
# FIXED HYPERPARAMETERS: C=1, max_iter=1000
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. FIXED MODEL CONFIGURATION
# ------------------------------------------------------------

lr_template = LogisticRegression(
    C=1.0,
    class_weight=None,
    max_iter=1000,
    random_state=42
)

# ------------------------------------------------------------
# 2. TRAIN AND EVALUATE ALL EXPERIMENTS
# ------------------------------------------------------------

results = []
fitted_models = {}
predictions = {}
reports = {}
confusion_matrices = {}

for exp_name in experiments:

    print("\n" + "=" * 65)
    print(exp_name)
    print("=" * 65)

    # Create a fresh model for each experiment
    model = LogisticRegression(
        C=1.0,
        class_weight=None,
        max_iter=1000,
        random_state=42
    )

    # Train
    print("Training Logistic Regression...")
    model.fit(
        X_train_experiments[exp_name],
        y_train
    )

    # Predict
    y_pred = model.predict(
        X_test_experiments[exp_name]
    )

    # Save fitted model and predictions
    fitted_models[exp_name] = model
    predictions[exp_name] = y_pred

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)

    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    results.append({
        "Experiment": exp_name,
        "Numeric Features": len(experiments[exp_name]),
        "Accuracy": accuracy,
        "Macro-F1": macro_f1,
        "Weighted-F1": weighted_f1
    })

    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Macro-F1    : {macro_f1:.4f}")
    print(f"Weighted-F1 : {weighted_f1:.4f}")

    # Classification report
    report = classification_report(
        y_test,
        y_pred,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0
    )

    reports[exp_name] = report

    print("\nClassification Report")
    print("-------------------------")

    print(classification_report(
        y_test,
        y_pred,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_,
        zero_division=0
    ))

    # Confusion matrix
    confusion_matrices[exp_name] = confusion_matrix(
        y_test,
        y_pred,
        labels=np.arange(len(label_encoder.classes_))
    )

# ------------------------------------------------------------
# 3. OVERALL COMPARISON TABLE
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Macro-F1",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 75)
print("EXPERIMENT COMPARISON: LOGISTIC REGRESSION, NO BALANCING")
print("=" * 75)

display(results_df.round(4))


E1_Text_Sentiment
Training Logistic Regression...
Accuracy    : 0.8434
Macro-F1    : 0.8247
Weighted-F1 : 0.8449

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age       0.95      0.97      0.96      2388
          ethnicity       0.97      0.98      0.98      2228
             gender       0.92      0.83      0.87      2243
  not_cyberbullying       0.59      0.61      0.60      1931
other_cyberbullying       0.57      0.60      0.59      1602
           religion       0.95      0.95      0.95      2389

           accuracy                           0.84     12781
          macro avg       0.83      0.82      0.82     12781
       weighted avg       0.85      0.84      0.84     12781


E2_Sentiment_Profanity_Behavior
Training Logistic Regression...
Accuracy    : 0.8453
Macro-F1    : 0.8272
Weighted-F1 : 0.8471

Classification Report
-------------------------
                     precision    recall  f1-sc

,Experiment,Numeric Features,Accuracy,Macro-F1,Weighted-F1
0,E4_Sentiment_All_Behavior,11,0.8456,0.8276,0.8474
1,E2_Sentiment_Profanity_Behavior,9,0.8453,0.8272,0.8471
2,E3_Sentiment_Mention_Hashtag,6,0.8448,0.8263,0.8463
3,E1_Text_Sentiment,4,0.8434,0.8247,0.8449
